# 📈 Hourly Time Series Forecasting with Timestral

This notebook is a translation of Rob Mulla's Kaggle notebook
[*Time Series forecasting with Prophet*](https://www.kaggle.com/code/robikscube/time-series-forecasting-with-prophet)
(Apache 2.0). It keeps the same data (**PJM East hourly electricity demand, 2002-2018**), the same train/test split
(train before 2015, test after) and the same metrics, but uses **Timestral**, Mistral AI's time-series foundation
model, instead of Prophet.

Every Prophet step is mapped to its Timestral equivalent:

| Prophet notebook | This notebook |
|---|---|
| `Prophet().fit(train)` | no fitting: `TimestralForecaster.from_pretrained(...)` |
| `model.predict(test)`: one forecast of the whole 2015-2018 test period | the same one-shot forecast (sec. 4), **plus** a realistic rolling week-ahead forecast (sec. 5) |
| `plot_components` | no components; average daily / weekly profiles of forecast vs actual instead |
| `Prophet(holidays=holiday_df)` | holiday indicator passed as a **known-future covariate** |
| cleaning: bad values set to NaN | NaN works the same way (or `observation_mask`) |

Reference numbers reported by the original notebook (Prophet, one forecast of the whole test set):
**MAE 5,181.8 MW, MAPE 16.5%** (16.5% with holidays, 16.5% after cleaning).

## Setup

In [ ]:
# Run once (uncomment):
%pip install -q "timestral @ git+https://github.com/mistralai/timestral@a534bdbd244d64f3d9d743f46c7bf6fc330ed4ff" \
                pandas matplotlib seaborn scikit-learn kagglehub

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import mean_squared_error, mean_absolute_error

from timestral import TimestralForecaster

plt.style.use("fivethirtyeight")  # For plots

MODEL_ID = "mistralai/Timestral-28M"   # Timestral-209M is the most accurate; 28M is ~5x faster on CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONTEXT_HOURS = 24 * 7 * 52            # history given to the model for rolling forecasts (1 year)
BATCH_SIZE = 16                        # series forecast per call in the rolling backtest

## Data

Hourly power consumption from PJM East (2002-2018), dataset
[`robikscube/hourly-energy-consumption`](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption).
On Kaggle it is attached as an input; elsewhere it is downloaded with `kagglehub`.

In [ ]:
def find_pjme_file() -> Path:
    for p in [Path("../input/PJME_hourly.csv"),
              Path("/kaggle/input/hourly-energy-consumption/PJME_hourly.csv"),
              Path("PJME_hourly.csv")]:
        if p.exists():
            return p
    import kagglehub
    return Path(kagglehub.dataset_download("robikscube/hourly-energy-consumption")) / "PJME_hourly.csv"


pjme = pd.read_csv(find_pjme_file(), index_col=[0], parse_dates=[0])  # index = Datetime

In [ ]:
# Color pallete for plotting
color_pal = ["#F8766D", "#D39200", "#93AA00", "#00BA38", "#00C19F", "#00B9E3", "#619CFF", "#DB72FB"]
pjme.plot(style=".", figsize=(15, 5), color=color_pal[0], title="PJM East")
plt.show()

### A regular hourly grid

This step has no counterpart in the Prophet notebook, because Prophet reads timestamps and does not care about
order or gaps. **Timestral reads an ordered sequence of equally spaced values and never sees the timestamps**, so
we must:

1. sort by time (the CSV is not in chronological order);
2. average the duplicated timestamps created by daylight-saving changes;
3. reindex to a complete hourly grid, leaving missing hours as NaN (Timestral handles NaN as "missing").

In [ ]:
pjme = pjme.sort_index()
n_dup = pjme.index.duplicated().sum()
pjme = pjme.groupby(level=0).mean()
full_index = pd.date_range(pjme.index.min(), pjme.index.max(), freq="h")
n_missing = len(full_index) - len(pjme)
pjme = pjme.reindex(full_index)
pjme.index.name = "Datetime"
pjme.index.freq = None  # keep pandas plots on plain matplotlib dates, so they mix with our own plots
print(f"{n_dup} duplicated hours averaged, {n_missing} missing hours set to NaN, {len(pjme):,} hourly steps")

## EDA

Same features as the original notebook, to see how demand depends on hour, day of week and season.

In [ ]:
def create_features(df, label=None):
    """Creates time series features from datetime index."""
    df = df.copy()
    df["date"] = df.index
    df["hour"] = df["date"].dt.hour
    df["dayofweek"] = df["date"].dt.dayofweek
    df["quarter"] = df["date"].dt.quarter
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["dayofyear"] = df["date"].dt.dayofyear
    df["dayofmonth"] = df["date"].dt.day
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)  # .dt.weekofyear was removed from pandas
    X = df[["hour", "dayofweek", "quarter", "month", "year", "dayofyear", "dayofmonth", "weekofyear"]]
    if label:
        return X, df[label]
    return X


X, y = create_features(pjme, label="PJME_MW")
features_and_target = pd.concat([X, y], axis=1)
features_and_target.head()

In [ ]:
sns.pairplot(features_and_target.dropna(), hue="hour", x_vars=["hour", "dayofweek", "year", "weekofyear"],
             y_vars="PJME_MW", height=5, plot_kws={"alpha": 0.15, "linewidth": 0})
plt.suptitle("Power Use MW by Hour, Day of Week, Year and Week of Year")
plt.show()

Power demand has strong **daily**, **weekly** and **yearly** patterns (winter and summer peaks). Prophet needs
these seasonalities declared as model terms; Timestral has to infer them from the history, which is why it should
see at least one full year of context.

## Train/Test Split

As in the original: train on data up to 1 Jan 2015, test on everything after.

In [ ]:
split_date = "2015-01-01"
pjme_train = pjme.loc[pjme.index <= split_date].copy()
pjme_test = pjme.loc[pjme.index > split_date].copy()

pjme_test.rename(columns={"PJME_MW": "TEST SET"}) \
    .join(pjme_train.rename(columns={"PJME_MW": "TRAINING SET"}), how="outer") \
    .plot(figsize=(15, 5), title="PJM East", style=".")
plt.show()

## 1. Load Timestral

The Prophet notebook calls `Prophet().fit(...)` on 12 years of hourly data (minutes of fitting). Timestral has no
training step: we load pretrained weights once.

In [ ]:
model = TimestralForecaster.from_pretrained(MODEL_ID, device=DEVICE)
print(f"{MODEL_ID} on {DEVICE} | max context {model.max_context:,} h "
      f"(~{model.max_context / 8766:.1f} years) | {len(model.quantile_levels)} quantiles")

A Prophet-like output: `ds`, `yhat`, `yhat_lower`, `yhat_upper` (80% interval, Prophet's default width).

In [ ]:
def to_prophet_frame(fc, index, series=0):
    return pd.DataFrame({"ds": index,
                         "yhat": fc.median[series, 0].cpu().numpy(),
                         "yhat_lower": fc.quantile(0.1)[series, 0].cpu().numpy(),
                         "yhat_upper": fc.quantile(0.9)[series, 0].cpu().numpy()})


def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates MAPE given y_true and y_pred"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    ok = ~np.isnan(y_true)
    return np.mean(np.abs((y_true[ok] - y_pred[ok]) / y_true[ok])) * 100


def report(name, y_true, y_pred):
    ok = ~np.isnan(np.asarray(y_true))
    yt, yp = np.asarray(y_true)[ok], np.asarray(y_pred)[ok]
    out = {"model": name, "MSE": mean_squared_error(yt, yp), "MAE": mean_absolute_error(yt, yp),
           "MAPE %": mean_absolute_percentage_error(yt, yp)}
    print(f"{name}: MSE {out['MSE']:,.0f} | MAE {out['MAE']:,.1f} | MAPE {out['MAPE %']:.2f}%")
    return out


scores = [{"model": "Prophet (original notebook)", "MSE": 43761675, "MAE": 5181.8, "MAPE %": 16.51}]

## 2. One-shot forecast of the whole test period (same setup as Prophet)

The original notebook forecasts **all ~31,000 test hours (3.6 years) in one go** from the training data. That is
what Prophet is designed for: its trend + seasonal curves can be extended indefinitely.

Timestral can do it too, but it is not its intended use: it decodes 1,024 hours per block and feeds each block's
median back as context, so beyond the first ~6 weeks it is forecasting from its own forecasts. We run it anyway,
for an apples-to-apples comparison with the same information as Prophet.

In [ ]:
H_test = len(pjme_test)
# The model reads at most `max_context` hours (32,768 ≈ 3.7 years); we cut the history to that window explicitly
history = pjme_train["PJME_MW"].to_numpy(dtype=np.float32)[-model.max_context:]
n_train, n_ctx = len(pjme_train), len(history)

t0 = time.time()
fc_oneshot = model.forecast(history, prediction_length=H_test)
print(f"{H_test:,} hours forecast in {time.time() - t0:.1f} s")

pjme_test_fcst = to_prophet_frame(fc_oneshot, pjme_test.index)
scores.append(report("Timestral one-shot", pjme_test["PJME_MW"], pjme_test_fcst["yhat"]))
pjme_test_fcst.head()

### Plot the forecast

In [ ]:
def plot_fcst(fcst, actual=None, title="", xbound=None, ylim=(0, 60000), ax=None):
    ax = ax or plt.subplots(figsize=(15, 5))[1]
    if actual is not None:
        ax.scatter(actual.index, actual["PJME_MW"], color="r", s=4, label="actual")
    ax.fill_between(fcst["ds"], fcst["yhat_lower"], fcst["yhat_upper"], color="#0072B2", alpha=0.2, lw=0,
                    label="80% interval")
    ax.plot(fcst["ds"], fcst["yhat"], color="#0072B2", lw=1, label="forecast")
    if xbound:
        ax.set_xlim(pd.Timestamp(xbound[0]), pd.Timestamp(xbound[1]))
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=13)
    ax.legend(loc="lower left", fontsize=9)
    return ax


ax = plt.subplots(figsize=(15, 5))[1]
last2y = pjme_train["PJME_MW"].tail(24 * 365 * 2)
ax.plot(last2y.index, last2y.values, ".", ms=1, color="k")
plot_fcst(pjme_test_fcst, title="Timestral one-shot forecast of 2015-2018 (from data up to 2015)", ax=ax)
plt.show()

### Compare Forecast to Actuals

In [ ]:
plot_fcst(pjme_test_fcst, pjme_test, title="One-shot forecast vs actuals")
plt.show()
plot_fcst(pjme_test_fcst, pjme_test, title="January 2015 Forecast vs Actuals", xbound=("2015-01-01", "2015-02-01"))
plt.show()
plot_fcst(pjme_test_fcst, pjme_test, title="First Week of January Forecast vs Actuals",
          xbound=("2015-01-01", "2015-01-08"))
plt.show()

### Forecast components → average profiles

Prophet's `plot_components` displays its own trend, daily, weekly and yearly terms. Timestral has no such terms,
so instead we check whether the forecast reproduces the **average patterns** of the real data: the daily profile,
the weekly profile and the monthly profile, forecast vs actual on the test period.

In [ ]:
def plot_profiles(fcst, actual, title):
    d = pd.DataFrame({"forecast": fcst.set_index("ds")["yhat"], "actual": actual["PJME_MW"]})
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, (label, key) in zip(axes, [("hour of day", d.index.hour), ("day of week (0 = Monday)", d.index.dayofweek),
                                       ("month", d.index.month)]):
        d.groupby(key).mean().plot(ax=ax, lw=2, color=["#0072B2", "r"])
        ax.set_xlabel(label)
        ax.set_ylabel("mean MW")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


plot_profiles(pjme_test_fcst, pjme_test, "One-shot forecast: average profiles vs actuals (2015-2018)")

## 3. Rolling week-ahead forecasts (how Timestral is meant to be used)

In operations nobody forecasts 3.6 years of hourly load in one shot: a grid operator forecasts the next day or
week, every day, with the latest data. That is the setting where a foundation model shines, because it needs no
refitting: it simply reads the most recent history.

Below, every Monday 00:00 of the test period we forecast the next **168 hours** from the previous year
(`CONTEXT_HOURS`). All the weekly forecasts are stitched together and scored on the same test set and metrics.
Note this is *not* the same information as Prophet's single forecast: each week uses the actuals up to that Monday.
Prophet could do this too, but would need ~190 refits on 12 years of hourly data.

In [ ]:
H = 168
series = pjme["PJME_MW"].to_numpy(dtype=np.float32)
pos = {ts: i for i, ts in enumerate(pjme.index)}
origins = [pos[t] for t in pd.date_range(pjme_test.index.min().normalize(), pjme.index.max(), freq="W-MON")
           if t in pos and pos[t] + H <= len(series)]
if pos[pjme_test.index.min()] not in origins:  # also cover the first test days (before the first Monday)
    origins = [pos[pjme_test.index.min()]] + origins


def rolling_forecast(model, origins, horizon, context, known=None):
    """Forecast `horizon` steps from each origin, in batches. `known`: optional (C, T) known-future covariates."""
    preds = {}
    for b in range(0, len(origins), BATCH_SIZE):
        chunk = origins[b:b + BATCH_SIZE]
        ctx = np.stack([series[o - context:o] for o in chunk])                      # (B, context)
        kw = {}
        if known is not None:
            kw["known_covariates"] = np.stack([known[:, o - context:o + horizon] for o in chunk])  # (B, C, ctx + H)
        fc = model.forecast(ctx, prediction_length=horizon, **kw)
        for j, o in enumerate(chunk):
            preds[o] = (fc.median[j, 0].cpu().numpy(), fc.quantile(0.1)[j, 0].cpu().numpy(),
                        fc.quantile(0.9)[j, 0].cpu().numpy())
    # stitch: each origin covers [o, next origin)
    yhat, lo, hi = (np.full(len(series), np.nan) for _ in range(3))
    for k, o in enumerate(origins):
        end = origins[k + 1] if k + 1 < len(origins) else o + horizon
        n = min(end - o, horizon)
        yhat[o:o + n], lo[o:o + n], hi[o:o + n] = preds[o][0][:n], preds[o][1][:n], preds[o][2][:n]
    test_mask = pjme.index > split_date
    return pd.DataFrame({"ds": pjme.index[test_mask], "yhat": yhat[test_mask], "yhat_lower": lo[test_mask],
                         "yhat_upper": hi[test_mask]})


t0 = time.time()
pjme_test_rolling = rolling_forecast(model, origins, H, CONTEXT_HOURS)
print(f"{len(origins)} week-ahead forecasts in {time.time() - t0:.0f} s")
covered = pjme_test_rolling["yhat"].notna().to_numpy()
scores.append(report("Timestral rolling week-ahead",
                     pjme_test["PJME_MW"].to_numpy()[covered], pjme_test_rolling["yhat"].to_numpy()[covered]))

In [ ]:
plot_fcst(pjme_test_rolling, pjme_test, title="January 2015: rolling week-ahead forecast vs actuals",
          xbound=("2015-01-01", "2015-02-01"))
plt.show()
plot_fcst(pjme_test_rolling, pjme_test, title="First Week of January: rolling week-ahead forecast vs actuals",
          xbound=("2015-01-01", "2015-01-08"))
plt.show()
plot_profiles(pjme_test_rolling, pjme_test, "Rolling week-ahead forecast: average profiles vs actuals")

## 4. Adding Holidays

Prophet takes a `holidays` DataFrame and learns one effect per holiday name. In Timestral, the equivalent is a
**known-future covariate**: a 0/1 series that is 1 during US federal holidays, given over the context *and* the
forecast horizon (holidays are known in advance). Series made only of 0/1 are treated as indicators by the model.

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar

cal = calendar()
holidays = cal.holidays(start=pjme.index.min(), end=pjme.index.max() + pd.Timedelta(days=1))
is_holiday = pd.Series(pjme.index.normalize().isin(holidays).astype(np.float32), index=pjme.index)
known_hol = is_holiday.to_numpy()[None, :]                                          # (1 covariate, T)

# Same holiday table as the original notebook, used later for per-holiday errors
holiday_df = pd.DataFrame({"ds": pjme.index[is_holiday.to_numpy() == 1], "holiday": "USFederalHoliday"})
print(f"{int(is_holiday.sum()):,} holiday hours, {holidays.size} holidays")
holiday_df.head()

### Predict with holidays (one-shot and rolling)

In [ ]:
fc_oneshot_hol = model.forecast(history, prediction_length=H_test,
                                known_covariates=known_hol[:, n_train - n_ctx:n_train + H_test][None])  # (1, 1, ctx + H)
pjme_test_fcst_with_hols = to_prophet_frame(fc_oneshot_hol, pjme_test.index)
scores.append(report("Timestral one-shot + holidays", pjme_test["PJME_MW"], pjme_test_fcst_with_hols["yhat"]))

pjme_test_rolling_hols = rolling_forecast(model, origins, H, CONTEXT_HOURS, known=known_hol)
scores.append(report("Timestral rolling week-ahead + holidays",
                     pjme_test["PJME_MW"].to_numpy()[covered], pjme_test_rolling_hols["yhat"].to_numpy()[covered]))

Note on the one-shot call: the known covariate must be exactly aligned with the context we pass (the last
`n_ctx` training hours) and extend over the whole horizon, hence the slice `[n_train - n_ctx, n_train + H_test)`.

### Compare models just for holiday dates: week of July 4th

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 9))
plot_fcst(pjme_test_rolling, pjme_test, title="Week of July 4th 2015: rolling, no holidays",
          xbound=("2015-07-01", "2015-07-08"), ax=axes[0])
plot_fcst(pjme_test_rolling_hols, pjme_test, title="Week of July 4th 2015: rolling, with holiday covariate",
          xbound=("2015-07-01", "2015-07-08"), ax=axes[1])
plt.tight_layout()
plt.show()

### Error for just the 4th of July

Small correction to the original: its query `Datetime >= 20160407` selects **7 April 2016**, not July 4th.
Here we select 4 July 2016 (Independence Day, a Monday that year).

In [ ]:
jul4 = (pjme_test.index >= "2016-07-04") & (pjme_test.index < "2016-07-05")
for name, f in [("rolling, no holidays", pjme_test_rolling), ("rolling, with holidays", pjme_test_rolling_hols),
                ("one-shot, no holidays", pjme_test_fcst), ("one-shot, with holidays", pjme_test_fcst_with_hols)]:
    print(f"July 4th 2016 MAE, {name:24s}: "
          f"{mean_absolute_error(pjme_test['PJME_MW'][jul4], f['yhat'].to_numpy()[jul4]):,.1f}")

### Error of all holidays, and by holiday

In [ ]:
hol_mask = pjme_test.index.isin(holiday_df["ds"]) & covered
y_hol = pjme_test["PJME_MW"].to_numpy()[hol_mask]
print("All holidays MAE, rolling without holidays:",
      f"{mean_absolute_error(y_hol, pjme_test_rolling['yhat'].to_numpy()[hol_mask]):,.1f}")
print("All holidays MAE, rolling with holidays   :",
      f"{mean_absolute_error(y_hol, pjme_test_rolling_hols['yhat'].to_numpy()[hol_mask]):,.1f}")

In [ ]:
rows = []
test_dates = pjme_test.index.normalize()
for day in sorted(set(test_dates[hol_mask])):
    m = (test_dates == day) & covered
    yt = pjme_test["PJME_MW"].to_numpy()[m]
    if np.isnan(yt).all():
        continue
    e0 = mean_absolute_error(yt, pjme_test_rolling["yhat"].to_numpy()[m])
    e1 = mean_absolute_error(yt, pjme_test_rolling_hols["yhat"].to_numpy()[m])
    rows.append({"holiday": f"{day:%B %d, %Y}", "MAE (no holiday covariate)": e0,
                 "MAE (holiday covariate)": e1, "Diff": e0 - e1})
by_holiday = pd.DataFrame(rows).set_index("holiday")
print(f"The holiday covariate helps on {(by_holiday.Diff > 0).sum()} of {len(by_holiday)} holidays")
by_holiday.round(1)

As the original notebook found with Prophet, lumping all federal holidays into a single indicator helps on some
days and hurts on others: Thanksgiving or Christmas do not affect demand like Columbus Day. A natural next step is a
**categorical** covariate with one code per holiday name, via `categorical_known_covariates` (codes from
`pandas.factorize`, target-encoded by the model).

### Plot error of each forecast

In [ ]:
for f, title in [(pjme_test_fcst, "One-shot forecast vs actuals"), (pjme_test_rolling_hols, "Rolling forecast vs actuals")]:
    ax = plt.subplots(figsize=(15, 5))[1]
    ax.plot(f["ds"], f["yhat"], ".", ms=2)
    ax.plot(pjme_test.index, pjme_test["PJME_MW"], ".", ms=2, alpha=0.2)
    plt.legend(["Forecast", "Actual"])
    plt.title(title)
    plt.show()

## 5. Data Cleaning

The original notebook spots abnormally low values in the training data (below 19,000 MW), caused by
**Hurricane Sandy** (29-30 October 2012, ~8 million customers without power).

In [ ]:
ax = pjme_train.plot(style=".", figsize=(15, 5), color=color_pal[5], title="PJME_MW with bad data highlighted",
                     xlim=("2012-10-29", "2012-11-01"), ylim=(10000, 31000))
pjme_train.query("PJME_MW < 19000").plot(style=".", figsize=(15, 5), color=color_pal[0], ax=ax)
ax.legend().set_visible(False)
plt.show()

Exactly as with Prophet, we replace the bad values with NaN; Timestral treats NaN as missing. (Alternatively, pass
`observation_mask=False` on those hours to keep the data but tell the model to ignore it.)

For the one-shot forecast, Sandy (2012) falls inside the 32,768-hour context the model reads before 2015, so this
is a real test. For the rolling forecasts (1 year of context, from 2015 on) Sandy is never in the context, so we
also look for outages in the **test** period, which do matter there.

In [ ]:
pjme_train["PJME_MW_clean"] = pjme_train["PJME_MW"]
pjme_train.loc[pjme_train["PJME_MW"] < 19000, "PJME_MW_clean"] = np.nan
print(f"{int((pjme_train['PJME_MW'] < 19000).sum())} training hours set to NaN")

fc_clean = model.forecast(pjme_train["PJME_MW_clean"].to_numpy(dtype=np.float32)[-n_ctx:], prediction_length=H_test)
pjme_test_fcst_clean = to_prophet_frame(fc_clean, pjme_test.index)
scores.append(report("Timestral one-shot, cleaned", pjme_test["PJME_MW"], pjme_test_fcst_clean["yhat"]))

low_test = pjme_test.query("PJME_MW < 19000")
print(f"{len(low_test)} test hours below 19,000 MW" + (f", e.g. {low_test.index[0]:%Y-%m-%d}" if len(low_test) else ""))

## Summary

In [ ]:
summary = pd.DataFrame(scores).set_index("model")
summary["MSE"] = summary["MSE"].round(0)
summary.round(2)

In [ ]:
ax = summary["MAPE %"].sort_values().plot(kind="barh", figsize=(12, 4.5),
                                          color=["#0072B2" if "Timestral" in i else "#7f7f7f"
                                                 for i in summary["MAPE %"].sort_values().index])
ax.set_xlabel("MAPE on the 2015-2018 test set (%) - lower is better")
plt.title("Prophet (original notebook) vs Timestral")
plt.show()

**How to read the results**

* **One-shot vs Prophet** is the fair comparison (same information: data up to 2015, one forecast for 3.6 years).
  Timestral was not built for 31,000-step horizons, so this is a stress test rather than its best use.
* **Rolling week-ahead** is the realistic use: each week uses the latest data. It is expected to be much more
  accurate, but it also uses more information than Prophet's single forecast, so it is not a like-for-like
  comparison. To compare Prophet in that setting you would have to refit it every week.
* **Holidays**: in Timestral they enter as covariates rather than model terms; as with Prophet, a single generic
  holiday flag is a crude feature, and per-holiday categories are worth trying.
* **Cleaning**: same idea as in Prophet (bad values → NaN), and with Timestral it only matters if the bad data is
  inside the context window the model actually reads.

Thanks for reading! Credit for the original analysis and narrative goes to Rob Mulla.

**References**
* Rob Mulla, [Time Series forecasting with Prophet](https://www.kaggle.com/code/robikscube/time-series-forecasting-with-prophet) (Kaggle, Apache 2.0)
* Dataset: [Hourly Energy Consumption (PJM)](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption)
* Mistral AI, [Introducing Timestral](https://mistral.ai/news/timestral/) and [mistralai/timestral](https://github.com/mistralai/timestral)